# LIGO Gravitational-Wave Parameter Estimation: Learn-By-Building Notebook

**Purpose:** build this entire project from scratch, one stage at a time,
following the natural real-world order: **explore the data → detect a
signal (classification) → estimate its properties with a simple baseline
(regression) → estimate its properties with full uncertainty (neural
posterior estimation) → test whether that uncertainty stays honest when
conditions change (calibration).**

Every stage has:
1. A **Concept** cell — the background/math, from first principles.
2. A **Task** cell — exactly what to build, with hints.
3. An **empty code cell** — where you write it yourself.

No solutions are included on purpose. If stuck, the original project chat
has working code for every step — but try first.

**Prerequisites:** Python, NumPy, matplotlib, basic calculus/linear algebra.
Zero prior knowledge of gravitational waves, Fourier analysis, hypothesis
testing, or probabilistic ML is assumed — all taught inline.

**Full stage list:**
0. Environment setup
0.5. Math foundations (Fourier, PSD, matched filtering, Bayes, hypothesis
     testing, calibration theory, normalizing flows)
1. What LIGO strain data is
2. EDA on raw noise
3. Time-domain plotting
4. Whitening
5. Bandpassing, spectrograms, and validating on a real event
6. Detection as hypothesis testing (classification)
7. Gathering real O3a/O4a noise
8. Simulating and injecting one merger
9. Scaling up to hundreds of injections
10. EDA on the injection dataset (+ checking your train/test split is fair)
11. Preprocessing into fixed-size training data
12. A point-estimate regression baseline
13. Neural posterior estimation (normalizing flows)
14. Calibration testing (SBC) across O3a vs O4a
15. A cheap drift-detection check, and wrap-up


## Stage 0 — Environment Setup

### Concept
This project needs a dedicated Python environment. Two reasons, learned the
hard way during development:
- `gwpy` and parts of `pycbc` pull in an authentication library chain
  (`scitokens` → `jwt` → `cryptography`) that's fragile on macOS and not
  needed for the public, unauthenticated GWOSC data we use.
- Mixing NumPy 2.x with certain PyTorch builds breaks `.numpy()` conversions
  with confusing errors.

We avoid GWpy entirely (use raw `numpy`/`scipy`/`h5py` instead) and pin
NumPy below 2.0.

### Task
Create a fresh conda environment and install exactly these packages, in
your terminal (not this notebook):

```bash
conda create -n ligo-poc python=3.11 -y
conda activate ligo-poc
pip install "numpy<2" scipy matplotlib boto3 h5py requests gwosc pycbc \
            sbi torch scikit-learn jupyter ipykernel
python -m ipykernel install --user --name ligo-poc --display-name "Python (ligo-poc)"
```

Select the `Python (ligo-poc)` kernel for this notebook before continuing.


In [1]:
# Sanity check -- should print versions with no import errors.
import numpy as np
import scipy
import matplotlib
import torch
import sklearn
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("torch:", torch.__version__)
print("sklearn:", sklearn.__version__)



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/alilordifar/opt/anaconda3/envs/ligo-poc/lib/python3.11/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/alilordifar/opt/anaconda3/envs/ligo-poc/lib/python3.11/runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "/Users/alilordifar/opt/anaconda3/envs/ligo-poc/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/alilordifar/opt/anaconda3/envs/ligo-poc/lib/python3.11/site-pa

numpy: 2.3.5
scipy: 1.16.3
torch: 2.2.2
sklearn: 1.9.1


## Stage 0.5 — Mathematical Foundations You'll Need

Read this once carefully; later stages point back to specific subsections
rather than re-deriving things. No code task here.

### A. The Fourier Transform

**The core idea:** any signal varying over time can equivalently be
described as a sum of pure sine waves at different frequencies. The
**Fourier Transform** converts a **time-domain** signal (value at each
moment) into its **frequency-domain** representation (how much of each
frequency it contains) — a change of basis, no information lost.

```
X(f) = ∫ x(t) e^(-i 2π f t) dt        (Fourier Transform)
x(t) = ∫ X(f) e^(i 2π f t) df         (Inverse Fourier Transform)
```

We use the **Discrete Fourier Transform (DFT)**, computed efficiently by
the **FFT** algorithm — exactly `np.fft.rfft`/`np.fft.irfft` (the "real"
variants, faster, since our signals are real-valued).

**Nyquist-Shannon Sampling Theorem:** a discretely-sampled signal can only
faithfully represent frequencies up to half the sample rate (the **Nyquist
frequency**). At 4096 Hz, that's 2048 Hz — comfortably above where GW
signals of interest live.

**Parseval's Theorem:** `∫|x(t)|² dt = ∫|X(f)|² df` — energy is conserved
across domains, which is why frequency-domain operations (like whitening)
correctly reflect real changes once transformed back.

### B. Power Spectral Density (PSD) and Wiener-Khinchin

A single FFT of noisy data is itself noisy; the *average* squared magnitude
across many segments converges to something meaningful: the **PSD**,
`S(f)` — how much power the signal has at each frequency, on average.
Formally (Wiener-Khinchin theorem), the PSD is the Fourier transform of the
signal's autocorrelation. In practice: **Welch's method**
(`scipy.signal.welch`) — split into overlapping chunks, FFT each, square,
average.

### C. Whitening, Formally

```
whitened(t) = IFFT[ FFT(x(t)) / sqrt(S(f)) ]
```
Divide by `sqrt(S(f))`, not `S(f)`, because PSD is a *squared*-magnitude
quantity — this correctly normalizes amplitude (not power) at each
frequency to unit variance.

### D. Convolution, Filtering, and Matched Filtering

**Convolution Theorem:** convolution in time equals multiplication in
frequency: `(x*h)(t) ⟷ X(f)·H(f)` — why frequency-domain filtering (and
whitening) works.

**Matched filtering**, the optimal way to detect a known signal shape in
noise, uses the noise-weighted inner product:
```
⟨d, h⟩ = 4 · Re ∫ [ d̃(f) h̃*(f) / S(f) ] df
```
**Optimal SNR** of a template: `SNR = sqrt(⟨h, h⟩)` — exactly what
`pycbc.filter.sigma()` computes.

### E. Probability & Statistics Foundations

**PDF/CDF:** `F(x) = P(X ≤ x) = ∫_{-∞}^{x} p(x') dx'`.

**Bayes' Theorem:**
```
p(θ | x) = p(x | θ) · p(θ) / p(x)
```
`p(θ)` = prior, `p(x|θ)` = likelihood, `p(θ|x)` = posterior. Simulation-based
inference exists because `p(x|θ)` is intractable for realistic GW noise —
we only need to *simulate* data given θ, never evaluate a likelihood.

**Probability Integral Transform:** if `X` has CDF `F`, then `F(X)` is
uniform on [0,1]. This is *why* SBC works: a correct posterior's rank
statistic (an estimate of the posterior's CDF at the true value) must be
uniformly distributed across many test cases.

### F. Change of Variables (Normalizing Flows)

A flow learns invertible `f` mapping a simple base distribution `z` to the
target posterior, `θ = f(z)`:
```
p_θ(θ) = p_z(f⁻¹(θ)) · |det ∂f⁻¹/∂θ|
```
A **Masked Autoregressive Flow (MAF)** makes this Jacobian triangular
(cheap determinant = product of diagonal entries), making training/sampling
tractable.

### G. Hypothesis Testing Fundamentals

This shows up in **four different places** in this project — it's worth
learning the general scaffolding once, here.

**The general recipe:**
1. State a **null hypothesis H₀** (the "boring," default explanation) and
   an **alternative H₁** (what you're trying to find evidence for).
2. Choose a **test statistic** — a number computed from data that behaves
   differently under H₀ than under H₁.
3. Compute a **p-value** — the probability of seeing a test statistic this
   extreme (or more) *if H₀ were actually true*.
4. Compare to a **significance level α** (commonly 0.05) — reject H₀ if
   p < α.

**Two types of error you can make:**
- **Type I error** (false positive): rejecting H₀ when it's actually true
  (e.g. claiming a detection when it's just noise).
- **Type II error** (false negative): failing to reject H₀ when H₁ is
  actually true (e.g. missing a real signal).
These trade off against each other — lowering your false-alarm rate
(stricter threshold) generally raises your miss rate, and vice versa. A
**ROC curve** plots this trade-off directly (true positive rate vs. false
positive rate as the decision threshold varies).

**Neyman-Pearson Lemma:** for a simple hypothesis test (H₀ vs. one specific
H₁), the likelihood ratio test is *provably* the most powerful test
possible for a given false-alarm rate. This is the theoretical reason
matched filtering (Stage 0.5.D) is the optimal way to detect a
known-shape signal in Gaussian noise — the matched-filter SNR *is* (up to
a monotonic transform) the Neyman-Pearson-optimal test statistic.

**One-sample vs. two-sample tests:** a one-sample test compares data
against a fully specified reference distribution (e.g. Stage 14's SBC
ranks vs. a known uniform distribution). A two-sample test compares two
*empirical* datasets against each other, asking if they could plausibly
come from the same distribution (e.g. Stage 15's O3a-vs-O4a noise
comparison). Both can use a KS test — the difference is what you're
comparing against.

**Multiple comparisons:** if you run several hypothesis tests at once
(e.g. one KS test per target parameter, 4 in total), your chance of at
least one false positive is higher than any single test's α. A
**Bonferroni correction** — using `α / (number of tests)` as your real
threshold — is the simplest fix, and matters directly in Stage 14.


## Stage 1 — What LIGO Data Actually Is

### Concept: strain, sample rate, and GPS time

LIGO measures whether space itself stretches or squeezes by an almost
unimaginably tiny amount. That measurement, recorded many times per second,
is called **strain** — just a long list of numbers, each tiny
(`~1e-19` to `1e-21`).

- **`sample_rate`**: measurements per second (4096 Hz — Nyquist = 2048 Hz,
  see Stage 0.5.A).
- **`dt`**: time between samples, `1/sample_rate`.
- **GPS time**: seconds since Jan 6, 1980 (not Unix epoch, Jan 1, 1970) —
  watch for this when timestamps look "off" by ~13 years or ~18 seconds.
- **Detector**: H1 (Hanford, WA) only in this project.

### Task
1. List H1 files in your S3 bucket
   (`signal-platform-dev-471112934830/raw/domain=ligo/source_id=H1/`).
2. Load one file's `.json` metadata sidecar and print it — identify the GPS
   field (some files have both a Unix-epoch field and a GPS field).
3. Load the matching `.npy` strain array. Print shape, dtype, min/max/mean/
   std. Sanity-check: shape should equal `sample_rate_hz * duration_sec`.


In [2]:
# Your code here: list S3 objects, load one segment + its metadata, print stats.



## Stage 2 — Exploratory Data Analysis on Raw Noise

### Concept: what to actually look for before doing anything else

Before writing any signal-processing code, get a feel for the raw data
itself — standard EDA practice, applied to a signal instead of a table.

- **Distribution of values**: plot a histogram of the strain values in your
  segment. Real detector noise is close to (but not exactly) Gaussian —
  worth checking directly rather than assuming.
- **Is it approximately Gaussian?** This is itself a hypothesis test (Stage
  0.5.G): `H₀: the values are normally distributed`. A **Kolmogorov-Smirnov
  test for normality** (`scipy.stats.kstest` against a fitted normal, or
  `scipy.stats.normaltest`) gives you a formal answer, not just a visual
  impression from the histogram.
- **Autocorrelation**: does a strain value at time `t` tell you anything
  about the value at `t + Δt`? Real noise (especially "colored" noise, per
  Stage 0.5.B) has structure here — plot the autocorrelation function
  (`np.correlate` or `statsmodels.tsa.stattools.acf`) for small lags and see
  how it decays.
- **Comparing multiple segments**: load 2-3 different noise segments (e.g.
  different days) and compare their basic stats (mean, std, min, max) side
  by side. Are they similar? This previews the "is O3a's noise different
  from O4a's noise" question from much later in the project (Stage 15) —
  worth noticing that even *within* one run, segment-to-segment variation
  exists.

### Task
1. Plot a histogram of your loaded segment's strain values.
2. Run a normality test (state your H₀/H₁, report the p-value, and say
   whether you reject H₀ at α=0.05).
3. Plot the autocorrelation function for the first ~50 lags. Does it decay
   quickly (suggesting mostly-random noise) or slowly (suggesting more
   structure/correlation)?
4. Load 2 more segments and compare basic stats in a small table.


In [ ]:
# Your code here: histogram, normality test (state H0/H1 + p-value), autocorrelation plot,
# and a comparison table across a few different noise segments.



## Stage 3 — Looking at Strain in the Time Domain

### Concept: why raw strain looks like noise, even with a real signal in it

Plotting raw strain over its full duration shows dense, uniform-looking
static. Even zooming into a confirmed detection's exact GPS time often
shows nothing obviously different by eye — expected, not a bug. Real
noise is "colored" (louder at low frequencies, Stage 0.5.B), which makes
it look smooth even with nothing interesting happening. Fixed in Stage 4.

### Task
1. Build a time axis: `times = gps_start + arange(N) * dt`.
2. Plot the full segment.
3. Look up a real event's merger GPS time (`gwosc.datasets.event_gps`).
   Zoom to ±1 second around it. Does a chirp jump out? (Maybe, for a loud
   event; probably not for a quiet one — note which case you used.)


In [ ]:
# Your code here: build the time axis, plot full segment, then zoom near a real event's merger time.



## Stage 4 — Frequency Domain & Whitening

### Concept: PSD, Welch's method, whitening formula

Applying Stage 0.5.A-C directly:
- Estimate `S(f)` via Welch's method (`scipy.signal.welch`).
- Whiten via `whitened = IFFT( FFT(strain) / sqrt(S(f)) )`.

Practical detail: Welch's output frequencies are coarser than a direct FFT
of your full segment — interpolate (`np.interp`) the PSD onto the FFT's
frequency grid before dividing.

### Task
1. Estimate the PSD via Welch's method.
2. Implement whitening via `np.fft.rfft`/`np.fft.irfft`, interpolating the
   PSD first.
3. Plot the whitened segment, zoomed near the same event as Stage 3.
   Compare to the raw zoom.


In [ ]:
# Your code here: estimate PSD via Welch, implement the whitening formula, plot the result.



## Stage 5 — Bandpassing and Seeing the Chirp

### Concept: bandpass filters, spectrograms, and why raw dB isn't enough

A **bandpass filter** (`scipy.signal.butter` + `filtfilt`) removes power
outside ~35-350 Hz. Per the Convolution Theorem (0.5.D), this is equivalent
to multiplying the spectrum by the filter's frequency response.

A **spectrogram** (time × frequency × loudness) is a series of short-time
FFTs. A plain spectrogram of pure noise has huge pixel-to-pixel variance —
compute **excess power** instead: divide each frequency row by its own
median across time, converting "raw loudness" into "how much louder than
usual, right now."

### Task
1. Bandpass your whitened segment.
2. Compute a spectrogram (`scipy.signal.spectrogram`) around a known
   event's merger.
3. Convert to excess power (per-row median normalization, dB), plot it.
4. **Validate on a loud event first** (e.g. GW150914, SNR ~24) — you should
   see an unmistakable upward-sweeping bright streak. If not, debug before
   proceeding — everything later depends on this working.


In [ ]:
# Your code here: bandpass, spectrogram, excess-power conversion, plot -- validate on GW150914 first.



## Stage 6 — Detection as Hypothesis Testing (Classification)

### Concept: "is there a signal here?" is a binary hypothesis test

Before ever estimating a merger's properties, real GW pipelines first have
to answer a yes/no question: is a signal present at all? Formally:
```
H₀: this window is pure detector noise
H₁: this window is noise + a real signal
```
Per the Neyman-Pearson Lemma (Stage 0.5.G), the matched-filter SNR is the
*provably optimal* test statistic for this exact decision (under Gaussian
noise) — this is why your SNR floor (used later, in Stage 9) is not an
arbitrary choice, it's a threshold on a principled hypothesis test.

This project frames detection as a small **binary classification**
problem: given a window of strain (or summary features computed from it),
predict "signal present" vs. "noise only." A simple, interpretable model —
**logistic regression** — is the natural first choice, and its decision
threshold is directly the same kind of Type I/Type II error trade-off
discussed in Stage 0.5.G.

### Task
1. Build a small labeled dataset: some windows containing an injected
   waveform (label 1) and some pure-noise windows (label 0) — reuse a
   single simple injection (like your Stage-8-to-come waveform generation,
   or borrow a quick fixed-parameter injection here) for the positive
   class, and plain noise segments for the negative class.
2. Compute a few simple summary features per window (e.g. matched-filter
   SNR against the segment's own PSD, peak excess power from Stage 5, total
   power in the 35-350 Hz band) rather than feeding raw strain directly.
3. Train a `sklearn.linear_model.LogisticRegression` on these features.
4. Plot a **ROC curve** (`sklearn.metrics.roc_curve`) and report the AUC.
   Pick a threshold and report your false-alarm rate and miss rate at that
   threshold — connect this back to Type I/Type II errors explicitly.


In [ ]:
# Your code here: build a labeled signal-vs-noise dataset, compute summary features,
# train logistic regression, plot an ROC curve, and report Type I/II error rates at a chosen threshold.



## Stage 7 — Gathering Real Noise for Two Observing Runs

### Concept: observing runs, why O3a vs O4a, and data quality flags

LIGO operates in **observing runs** (O1, O2, O3a/O3b, O4a/...) separated by
upgrades that measurably change the detector's noise PSD (Stage 0.5.B).

This project compares **O3a** (2019) vs. **O4a** (2023) — one clean,
well-documented upgrade cycle, unlike O1-vs-O2 (too similar) or O2-vs-O4a
(two upgrade cycles confounded).

Clean noise requires: segments spread across the run (not one day), ≥1hr
buffer from any cataloged event, and passing GWOSC's per-second **data
quality bitmask** (we check CBC-relevant bits, not burst-search bits).

### Task
1. `gwosc.datasets.run_segment("O3a")` for GPS boundaries.
2. `gwosc.datasets.find_datasets(type="events", segment=...)` +
   `event_gps()` for every cataloged event's GPS time.
3. Generate spread-out candidate start-times, reject any within 1hr of an
   event.
4. For each candidate: fetch via `gwosc.locate.get_urls` + `requests` +
   `h5py`, check DQ bits (`DATA`, `CBC_CAT1/2/3`), reject NaNs/implausible
   std. Keep going until 8 clean segments.
5. **Checkpoint to disk as you go** — this step is slow and *will* get
   interrupted; don't redo finished work.
6. Repeat for O4a.


In [ ]:
# Your code here: get O3a run boundaries + events, generate candidates, fetch + quality-check 8 clean segments.
# Then repeat for O4a. Remember to checkpoint (save to disk, skip if already done).



## Stage 8 — Simulating a Merger and Hiding It in Real Noise

### Concept: why simulate, BBH parameters, matched-filter SNR

Real confirmed mergers are too rare to train on. Instead: simulate a
waveform (`IMRPhenomD` via `pycbc`) and add it onto real noise, giving
unlimited examples with exact known ground truth.

- **Chirp mass**: `M_c = (m1*m2)^(3/5) / (m1+m2)^(1/5)` — controls sweep
  rate directly, via `df/dt ∝ M_c^(5/3) f^(11/3)`.
- **Mass ratio**: `q = m2/m1` (`m1 >= m2` by convention).
- **Effective spin**: `chi_eff = (m1·chi1 + m2·chi2)/(m1+m2)`.
- **Distance**: strain amplitude falls as `1/distance`.

**Matched-filter SNR** (Stage 0.5.D): `SNR = sqrt(⟨h, h⟩)`, via
`pycbc.filter.sigma()`. Real detections require SNR > 8 — the same
detection threshold from Stage 6, now applied as a filter on which
injections to keep.

**Bug to watch for:** PyCBC's waveform array is not merger-aligned at its
last sample (includes ringdown/padding). Always align using
`peak_idx = np.argmax(np.abs(waveform))`, never the array's length.

### Task
1. Generate one waveform (own masses/spins, a *realistic* distance — not
   the 1 Mpc default).
2. Find its true peak (merger) index.
3. Inject it into a Stage 7 noise segment, aligned by peak index.
4. Compute its matched-filter SNR against that segment's PSD.
5. Whiten + bandpass + excess-power-spectrogram it (reuse Stages 4-5) and
   confirm you can see it.


In [ ]:
# Your code here: generate one waveform, align by its peak, inject into real noise,
# compute matched-filter SNR, and visually confirm it via whitening + excess-power spectrogram.



## Stage 9 — Generating Hundreds of Injections

### Concept: priors, SNR rejection

The **prior** `p(θ)` (Stage 0.5.E) is Bilby's standard default BBH prior:
masses uniform 5-100 M☉, aligned spin 0-0.8, distance uniform-in-comoving-
volume up to 2000 Mpc (tighter than Bilby's 5000 Mpc default, to avoid
wasting budget on undetectable injections).

We reject injections with SNR < 8 (same detection threshold from Stages 6
and 8) — expect a real, sometimes-high rejection rate, not a bug.

**Memory lesson:** don't store the full noise segment per injection —
tens of GB across hundreds of injections will crash your kernel. Store
only a small buffer around merger, sized to the waveform's actual length
(low chirp mass → slower frequency sweep → longer waveform, per Stage 8's
`df/dt` relation).

### Task
1. Write `draw_bbh_params()` — one random draw from the prior above.
2. Write `generate_injection()` — waveform → SNR check (reject <8) →
   inject into a small, dynamically-sized buffer → return buffer + true
   params + SNR.
3. Loop to ~800 accepted injections for O3a (random noise segment + random
   merger placement each time). Track/print rejection rate.
4. Checkpoint to disk; guard against re-running if a valid checkpoint
   exists (and against a corrupt/empty checkpoint file).
5. Repeat for O4a. (Prediction: O4a should show a *lower* rejection rate,
   since it's more sensitive — check this.)


In [ ]:
# Your code here: draw_bbh_params(), generate_injection() with SNR-floor rejection and a
# small dynamically-sized buffer, then loop to ~800 accepted injections each for O3a and O4a,
# with checkpointing.



## Stage 10 — EDA on Your Injection Dataset (and a Fairness Check)

### Concept: what to look at before training anything

Now that you have hundreds of injections with known parameters, explore
them like any other dataset before modeling:
- **Distributions**: histogram each target parameter (chirp mass, mass
  ratio, chi_eff, distance) and the SNR. Do they look like what the prior
  should produce? Is SNR skewed toward low values (expected, since low-SNR
  injections are the ones sitting right above the rejection floor)?
- **Correlations**: does SNR correlate with distance (it should — closer
  = louder) or with chirp mass? A simple correlation matrix or pairplot is
  enough.
- **A real hypothesis test you should actually run**: before trusting any
  later result, check that your random train/test split didn't
  accidentally produce meaningfully different parameter distributions in
  each half. This is a genuine two-sample hypothesis test (Stage 0.5.G):
  ```
  H₀: the training set's chirp mass distribution and the held-out test
      set's chirp mass distribution come from the same underlying
      distribution
  ```
  A **two-sample KS test** (`scipy.stats.ks_2samp`) or a **Mann-Whitney U
  test** (`scipy.stats.mannwhitneyu`) — a non-parametric test that doesn't
  assume normality, often a safer default than a t-test here — gives you a
  p-value for this directly.

### Task
1. Plot histograms for chirp mass, mass ratio, chi_eff, distance, and SNR
   across your O3a injections.
2. Compute a correlation matrix between SNR and the 4 target parameters.
3. Run a two-sample test (KS or Mann-Whitney) comparing your training
   split's chirp mass distribution against your held-out test split's
   (you'll finalize the actual split in Stage 11 — for now, do a quick
   random 85/15 split just for this check). State H₀/H₁ and your
   conclusion at α=0.05.


In [ ]:
# Your code here: histograms of target parameters + SNR, a correlation matrix,
# and a two-sample hypothesis test comparing a train/test split's chirp mass distributions.



## Stage 11 — Turning Injections into Fixed-Size Training Data

### Concept: fixed windows, per-injection whitening, real train/test split

Every training example must be the same shape: crop to **2 seconds before +
2 seconds after merger** = 16,384 numbers at 4096 Hz, regardless of the
original buffer/waveform length.

Whiten each injection using its **own** originating noise segment's PSD
(not shared/pooled) — matches real practice, avoids leaking information
between differently-sourced injections.

Hold out ~50 O3a injections for testing (never touched in training). All
O4a injections are test-only — we never train on O4a, since the goal is
testing generalization to noise the model never saw.

### Task
1. `compute_derived_params()`: raw params → (chirp mass, mass ratio,
   chi_eff, distance) using Stage 8's formulas.
2. `preprocess_injection()`: whiten (own segment's PSD) + bandpass + crop
   to the fixed window.
3. Apply to all O3a injections (split: ~750 train / ~50 held-out) and all
   O4a injections (test-only).
4. Print final shapes, e.g. `X_train.shape = (750, 16384)`,
   `theta_train.shape = (750, 4)`.


In [ ]:
# Your code here: compute_derived_params(), preprocess_injection() (whiten+bandpass+crop),
# then build X_train/theta_train, X_test_o3a/theta_test_o3a, X_test_o4a/theta_test_o4a.



## Stage 12 — A Point-Estimate Regression Baseline

### Concept: what a "normal" ML model gives you, and what it can't

Before jumping to a full posterior-estimating neural network, it's worth
seeing what a plain, boring regression model can do on this same task —
this sets up exactly why the more complex approach (Stage 13) is worth its
complexity.

A **linear regression** (or a small multi-layer perceptron, if you want a
slightly stronger baseline) trained on `(X, θ)` pairs gives you **one
number per parameter** — a point estimate, with no sense of how confident
it should be. You can measure its quality with standard regression
metrics: **MAE** (mean absolute error) and **RMSE** (root mean squared
error), per parameter.

**A practical/statistical note:** your raw strain input has 16,384
dimensions, but you only have ~750 training examples — far too few
examples relative to dimensions for plain linear regression on raw strain
to behave sensibly (severe overfitting/multicollinearity risk). Two honest
options, both worth trying and comparing:
- Feed the model **summary features** instead of raw strain (e.g. matched-
  filter SNR, peak excess-power frequency, total in-band power — similar to
  what you computed in Stage 6).
- Feed it raw strain anyway, observe it perform poorly, and treat that
  poor performance itself as the lesson in why dimensionality matters.

### Task
1. Build a small feature set per injection (reuse ideas from Stage 6).
2. Train `sklearn.linear_model.LinearRegression` (one model per target
   parameter, or `MultiOutputRegressor`) on your O3a training set.
3. Evaluate MAE/RMSE per parameter on the O3a held-out test set.
4. Write a short markdown note: what does this baseline get right, and
   what can it fundamentally never tell you, no matter how good its MAE
   gets? (Hint: think about what "confidence" would even mean for this
   model's output.)


In [ ]:
# Your code here: build features, train a linear regression baseline per target parameter,
# report MAE/RMSE on the O3a held-out set.



## Stage 13 — Training a Model That Outputs Uncertainty, Not Just a Guess

### Concept: Bayesian inference without a likelihood, and normalizing flows

Recall Bayes' theorem (0.5.E): `p(θ|x) ∝ p(x|θ)·p(θ)`. Classical Bayesian
PE (MCMC, nested sampling) requires evaluating `p(x|θ)` directly and
repeatedly — expensive, and only exact under idealized Gaussian-noise
assumptions that real detector noise doesn't perfectly satisfy.

**Simulation-Based Inference (SBI)** instead only needs a **simulator**
(exactly Stages 8-9: draw θ, simulate, inject into real noise) — no
explicit likelihood needed. **Neural Posterior Estimation (NPE)** trains a
network directly on `(θ, x)` pairs to approximate `p(θ|x)`, enabling near-
instant posterior sampling afterward.

A **normalizing flow** learns invertible `f`, `θ = f(z)`, via the change-of-
variables formula (0.5.F). We use a **Masked Autoregressive Flow (MAF)** —
its triangular Jacobian makes training/sampling tractable. Training
maximizes `log p_θ(θ_true | x)` averaged over the training set — this is
`sbi`'s `SNPE.train()`.

**Contrast with Stage 12:** this model outputs a full distribution, not
one number — directly enabling the calibration test in Stage 14, which
Stage 12's regression baseline structurally cannot support.

### Task
1. Convert `X_train`/`theta_train` to PyTorch tensors.
2. Define `sbi.utils.BoxUniform` matching your *actual* `theta_train`
   ranges (transformed-parameter space, not the raw Stage 9 prior).
3. Train `sbi.inference.SNPE`, build the posterior.
4. Checkpoint the trained posterior to disk.


In [ ]:
# Your code here: convert to tensors, define the BoxUniform prior, train SNPE, save the posterior.



## Stage 14 — Testing Whether the Model's Confidence Is Honest

### Concept: calibration, SBC, and multiple-comparisons correction

A model can be accurate on average while dishonest about its own
uncertainty. **Calibration** tests whether stated confidence matches
reality — accuracy alone can't catch this.

**SBC**, via the Probability Integral Transform (0.5.E): for each held-out
example, find the true value's **rank** among posterior samples. A correct
posterior implies these ranks are uniformly distributed across many test
examples (0.5.E's theorem, applied). A **KS test** (0.5.G, one-sample:
against a known uniform reference) gives a formal statistic.

**Multiple comparisons (0.5.G):** you're running this test **4 times**
(once per parameter) — at α=0.05 each, your overall false-positive risk
across all 4 is higher than 5%. Apply a **Bonferroni correction**: use
α/4 ≈ 0.0125 as your real per-test threshold before calling any single
parameter "significantly miscalibrated."

**Sampling hazard:** posterior rejection-sampling can hang if proposals
rarely land inside the prior for a given input (a real risk we hit — lost
a whole night to it once). Always pass `max_sampling_time`, and exclude
(never zero-fill) any example that fails.

### Task
1. Run SBC on your ~50 O3a held-out examples (with a sampling-time cap).
2. KS-test the ranks per parameter — apply the Bonferroni-corrected
   threshold when deciding "calibrated" vs. "not," not the raw α=0.05.
3. Repeat identically on O4a. Compare KS statistics (not just p-values —
   different sample sizes make p-values misleading on their own) between
   O3a and O4a, per parameter.
4. If O4a's test set is much larger than O3a's, also try a few repeated
   random subsamples of O4a matched to O3a's size, before concluding
   anything from small differences.


In [ ]:
# Your code here: run SBC on O3a held-out (baseline), then on O4a (cross-run test),
# compute KS-test statistics per parameter for both (with Bonferroni correction), and compare.



## Stage 15 — A Cheap Early-Warning Signal, and Wrapping Up

### Concept: a two-sample test on the noise itself

SBC (Stage 14) is expensive — it needs a trained model and hundreds of
posterior-sampling calls. A much cheaper, complementary question: do the
two runs' noise PSDs actually look different, directly? This is a genuine
**two-sample hypothesis test** (0.5.G):
```
H₀: O3a and O4a noise PSDs (in the signal band) come from the same
    underlying distribution
H₁: they don't
```
Note this is structurally different from Stage 14's one-sample test
(comparing observed ranks against a *known* uniform reference) — here
you're comparing two *empirical* samples against each other.

This doesn't replace SBC — "the input distribution changed" (cheaply
measurable here) and "the model's behavior changed as a result" (only
measurable via the expensive Stage 14 test) are related but genuinely
different questions.

### Task
1. Compute the average PSD (Welch, averaged across segments) for O3a and
   O4a. Plot both on log-log axes.
2. Run a two-sample KS test (`scipy.stats.ks_2samp`) comparing the two ASD
   curves in the 20-500 Hz band. State H₀/H₁ and your conclusion.
3. In a new markdown cell, write your overall findings: does Stage 14's
   SBC comparison show calibration degrading O3a→O4a? Does that match or
   contradict what this cheap drift check would have predicted before you
   ever trained anything? What surprised you most across the whole
   project?


In [ ]:
# Your code here: compute + plot average O3a/O4a PSDs, run a two-sample KS test between them,
# and write your findings in a markdown cell below.



### Your findings (replace this cell with your own writeup)

_Write here: what did you find at each stage -- detection, the regression
baseline's limits, NPE's calibration on O3a, and on O4a? Does calibration
hold up across the noise shift? What would you investigate next if you had
another week?_
